In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    BASE_DIR,
    TRAIN_PATH_STAGE1,
    TEST_PATH_STAGE1,
    CRAWLED_ABSTRACT_PATH,
    MERGED_TRAIN_PATH,
    MERGED_TEST_PATH,
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH
)

from src.utils.utils import load_csv, save_csv, preview_df

from src.preprocess.preprocess import (
    DataMerger,
    MissingHandler,
    AuthorNormalizer,
    TextCleaner
)

import pandas as pd

print("✔ Imports ready")

✔ Imports ready


In [3]:
train_df = load_csv(TRAIN_PATH_STAGE1)
test_df = load_csv(TEST_PATH_STAGE1)
crawl_df = load_csv(CRAWLED_ABSTRACT_PATH)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Crawl:", crawl_df.shape)

preview_df(train_df)

Train: (510, 7)
Test: (86, 6)
Crawl: (543, 5)
Shape: (510, 7)
Columns: ['title', 'venue', 'year', 'authors', 'doi', 'Label', 'id']


,title,venue,year,authors,doi,Label,id
0,Tabled CLP for Reasoning Over Stream Data.,iclp,2016,"Edmond Jajaga, Lule Ahmedi",10.1109/icsc.2017.64,1,299
1,XAI-LAW: A Logic Programming Tool for Modeling...,iclp,2025,"Agostino Dovier, Talissa Dreossi, Andrea Formi...",10.4204/eptcs.439.28,5,92
2,Workshop Proceedings of the 40th International...,iclp,2024,"Esteban Guerrero, Juan Carlos Nieves",10.1007/978-3-031-74209-5_21,1,311
3,Probabilistic Active Goal Recognition.,kr,2025,"Chenyuan Zhang, Cristian Rojas Cardenas, Hamid...",10.24963/kr.2025/85,5,136
4,Heuristic Strategies for Accelerating Multi-Ag...,kr,2024,"Biqing Fang, Fangzhen Lin",10.24963/kr.2024/32,1,41


In [4]:
merger = DataMerger()

train_merged = merger.transform(train_df, crawl_df)
test_merged = merger.transform(test_df, crawl_df)

print("Train merged:", train_merged.shape)
print("Test merged:", test_merged.shape)

preview_df(train_merged)

Train merged: (510, 10)
Test merged: (86, 9)
Shape: (510, 10)
Columns: ['title', 'venue', 'year', 'authors', 'doi', 'Label', 'id', 'abstract', 'crawled_authors', 'source']


,title,venue,year,authors,doi,Label,id,abstract,crawled_authors,source
0,Tabled CLP for Reasoning Over Stream Data.,iclp,2016,"Edmond Jajaga, Lule Ahmedi",10.1109/icsc.2017.64,1,299,Semantic technologies have been extensively us...,"Edmond Jajaga, Lule Ahmedi",OpenAlexCrawler
1,XAI-LAW: A Logic Programming Tool for Modeling...,iclp,2025,"Agostino Dovier, Talissa Dreossi, Andrea Formi...",10.4204/eptcs.439.28,5,92,We propose an approach to model articles of th...,"A. Dovier, Talissa Dreossi, A. Formisano, Bene...",SemanticScholarCrawler
2,Workshop Proceedings of the 40th International...,iclp,2024,"Esteban Guerrero, Juan Carlos Nieves",10.1007/978-3-031-74209-5_21,1,311,"Traditionally, in the argumentation theory lit...",NaN,DOIResolverCrawler
3,Probabilistic Active Goal Recognition.,kr,2025,"Chenyuan Zhang, Cristian Rojas Cardenas, Hamid...",10.24963/kr.2025/85,5,136,"In multi-agent environments, effective interac...","Chenyuan Zhang, Cristian Rojas Cardenas, Hamid...",CrossRefCrawler
4,Heuristic Strategies for Accelerating Multi-Ag...,kr,2024,"Biqing Fang, Fangzhen Lin",10.24963/kr.2024/32,1,41,Multi-agent epistemic planning (MEP) is about ...,"Biqing Fang, Fangzhen Lin",SemanticScholarCrawler


In [13]:
drop_cols = ["crawled_authors", "source"]

train_merged = train_merged.drop(

    columns=[c for c in drop_cols if c in train_merged.columns],

    errors="ignore"

)

test_merged = test_merged.drop(

    columns=[c for c in drop_cols if c in test_merged.columns],

    errors="ignore"

)

# =========================

# SAVE

# =========================

save_csv(train_merged, MERGED_TRAIN_PATH)

save_csv(test_merged, MERGED_TEST_PATH)

print("✔ Merged data saved to interim/")

Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/merged_train.csv
Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/merged_test.csv
✔ Merged data saved to interim/


In [14]:
missing = MissingHandler()

train_clean = missing.transform(train_merged)
test_clean = missing.transform(test_merged)

print("After missing handling:")
print(train_clean.isna().sum())

After missing handling:
title       0
venue       0
year        0
authors     0
doi         0
Label       0
id          0
abstract    0
dtype: int64


In [15]:
author_norm = AuthorNormalizer()

train_clean = author_norm.transform(train_clean)
test_clean = author_norm.transform(test_clean)

preview_df(train_clean)

Shape: (510, 8)
Columns: ['title', 'venue', 'year', 'authors', 'doi', 'Label', 'id', 'abstract']


,title,venue,year,authors,doi,Label,id,abstract
0,Tabled CLP for Reasoning Over Stream Data.,iclp,2016,"edmond jajaga, lule ahmedi",10.1109/icsc.2017.64,1,299,Semantic technologies have been extensively us...
1,XAI-LAW: A Logic Programming Tool for Modeling...,iclp,2025,"agostino dovier, talissa dreossi, andrea formi...",10.4204/eptcs.439.28,5,92,We propose an approach to model articles of th...
2,Workshop Proceedings of the 40th International...,iclp,2024,"esteban guerrero, juan carlos nieves",10.1007/978-3-031-74209-5_21,1,311,"Traditionally, in the argumentation theory lit..."
3,Probabilistic Active Goal Recognition.,kr,2025,"chenyuan zhang, cristian rojas cardenas, hamid...",10.24963/kr.2025/85,5,136,"In multi-agent environments, effective interac..."
4,Heuristic Strategies for Accelerating Multi-Ag...,kr,2024,"biqing fang, fangzhen lin",10.24963/kr.2024/32,1,41,Multi-agent epistemic planning (MEP) is about ...


In [16]:
text_cleaner = TextCleaner()

train_clean = text_cleaner.transform(train_clean)
test_clean = text_cleaner.transform(test_clean)

preview_df(train_clean)

Shape: (510, 8)
Columns: ['title', 'venue', 'year', 'authors', 'doi', 'Label', 'id', 'abstract']


,title,venue,year,authors,doi,Label,id,abstract
0,tabled clp for reasoning over stream data,iclp,2016,"edmond jajaga, lule ahmedi",10.1109/icsc.2017.64,1,299,semantic technologies have been extensively us...
1,xai-law: a logic programming tool for modeling...,iclp,2025,"agostino dovier, talissa dreossi, andrea formi...",10.4204/eptcs.439.28,5,92,we propose an approach to model articles of th...
2,workshop proceedings of the 40th international...,iclp,2024,"esteban guerrero, juan carlos nieves",10.1007/978-3-031-74209-5_21,1,311,traditionally in the argumentation theory lite...
3,probabilistic active goal recognition,kr,2025,"chenyuan zhang, cristian rojas cardenas, hamid...",10.24963/kr.2025/85,5,136,in multi-agent environments effective interact...
4,heuristic strategies for accelerating multi-ag...,kr,2024,"biqing fang, fangzhen lin",10.24963/kr.2024/32,1,41,multi-agent epistemic planning mep is about ac...


In [17]:
save_csv(train_clean, CLEANED_TRAIN_PATH)
save_csv(test_clean, CLEANED_TEST_PATH)

print("✔ Cleaned data saved")

Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/cleaned_train.csv
Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/cleaned_test.csv
✔ Cleaned data saved


In [18]:
print("FINAL SHAPES:")
print("Train:", train_clean.shape)
print("Test:", test_clean.shape)

train_clean.head()

FINAL SHAPES:
Train: (510, 8)
Test: (86, 7)


,title,venue,year,authors,doi,Label,id,abstract
0,tabled clp for reasoning over stream data,iclp,2016,"edmond jajaga, lule ahmedi",10.1109/icsc.2017.64,1,299,semantic technologies have been extensively us...
1,xai-law: a logic programming tool for modeling...,iclp,2025,"agostino dovier, talissa dreossi, andrea formi...",10.4204/eptcs.439.28,5,92,we propose an approach to model articles of th...
2,workshop proceedings of the 40th international...,iclp,2024,"esteban guerrero, juan carlos nieves",10.1007/978-3-031-74209-5_21,1,311,traditionally in the argumentation theory lite...
3,probabilistic active goal recognition,kr,2025,"chenyuan zhang, cristian rojas cardenas, hamid...",10.24963/kr.2025/85,5,136,in multi-agent environments effective interact...
4,heuristic strategies for accelerating multi-ag...,kr,2024,"biqing fang, fangzhen lin",10.24963/kr.2024/32,1,41,multi-agent epistemic planning mep is about ac...


In [19]:
# =========================
# MISSING VALUE ANALYSIS (REALISTIC)
# =========================

def missing_report(df, name="dataset"):
    print(f"\n===== MISSING REPORT: {name} =====")
    
    report_rows = []

    for col in df.columns:
        if df[col].dtype == "object":
            missing_count = (df[col].isna() | (df[col].astype(str).str.strip() == "")).sum()
        else:
            missing_count = df[col].isna().sum()

        missing_ratio = missing_count / len(df) * 100

        report_rows.append((col, missing_count, missing_ratio))

    report = pd.DataFrame(
        report_rows,
        columns=["column", "missing_count", "missing_ratio_%"]
    )

    report = report[report["missing_count"] > 0].sort_values(
        by="missing_count",
        ascending=False
    )

    if len(report) == 0:
        print("No missing values 🎉 (including empty strings)")
    else:
        print(report)

    return report


# =========================
# RUN
# =========================
train_missing_report = missing_report(train_df, "RAW TRAIN")
test_missing_report = missing_report(test_df, "RAW TEST")

train_clean_missing_report = missing_report(train_clean, "CLEAN TRAIN")
test_clean_missing_report = missing_report(test_clean, "CLEAN TEST")


===== MISSING REPORT: RAW TRAIN =====
    column  missing_count  missing_ratio_%
3  authors             52        10.196078

===== MISSING REPORT: RAW TEST =====
    column  missing_count  missing_ratio_%
3  authors              8         9.302326

===== MISSING REPORT: CLEAN TRAIN =====
No missing values 🎉 (including empty strings)

===== MISSING REPORT: CLEAN TEST =====
No missing values 🎉 (including empty strings)
